# Sudoku Solving: A\*, Hill Climbing and Backtracking

**Course:** Artificial Intelligence | **Dataset:** [Kaggle Sudoku (rohanrao)](https://www.kaggle.com/datasets/rohanrao/sudoku?select=sudoku.csv)

Our faculty gave us one dataset and asked us to apply different algorithms to it and explain which one is the best. The dataset is a huge collection of Sudoku puzzles, and every puzzle comes with its correct solution, so we can check each algorithm's answer instead of just trusting it.

The three algorithms we compare:

1. **Backtracking** (Constraint Satisfaction Problem search)
2. **A\* Search** (Informed search)
3. **Hill Climbing** (Local search)

The notebook runs all three on the same puzzles, under the same time limit, and then compares them using graphs. The last section picks the best one from the numbers.

## 1. Setup

This cell downloads the dataset from Kaggle. It needs your Kaggle token saved in Colab secrets under the name `KAGGLE_API_TOKEN`. If you already have `sudoku.csv` in the Colab files panel, you can skip it.

In [ ]:
from google.colab import userdata
import os
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
token_path = os.path.expanduser('~/.kaggle/access_token')
with open(token_path, 'w') as f:
    f.write(userdata.get('KAGGLE_API_TOKEN'))
os.chmod(token_path, 0o600)
!kaggle datasets download -d rohanrao/sudoku
!unzip -q -o sudoku.zip
print('Dataset ready')

## 2. Load the data and pick puzzles to test

The full file has millions of puzzles, so we read only the first 100,000 rows and draw a random sample from them. The sample is sorted by the number of given clues and split into three equal groups. Fewer clues means a harder puzzle, so the groups are called Hard, Medium and Easy.

Two settings control the experiment:

* `N_PUZZLES`: how many puzzles to test. Lower it if you want a faster run.
* `TIME_LIMIT`: seconds each algorithm gets per puzzle. If it runs out, that puzzle counts as unsolved.

In [ ]:
import time, random, heapq, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', context='talk')
palette = {'Backtracking': '#264653', 'A*': '#2a9d8f', 'Hill Climbing': '#e76f51', 'Backtracking + MRV': '#e9c46a'}
N_PUZZLES = 30
TIME_LIMIT = 5
pool = pd.read_csv('sudoku.csv', nrows=100000, dtype={'puzzle': str, 'solution': str})
pool['clues'] = pool['puzzle'].str.count(r'[1-9]')
sample = pool.sample(N_PUZZLES, random_state=42).sort_values('clues').reset_index(drop=True)
sample['level'] = pd.qcut(sample.index, 3, labels=['Hard', 'Medium', 'Easy'])
print(sample.groupby('level', observed=True)['clues'].agg(['min', 'max', 'mean']).round(1))

In [ ]:
def string_to_board(text):
    return [[int(ch) if ch.isdigit() else 0 for ch in text[i:i + 9]] for i in range(0, 81, 9)]
def board_to_string(board):
    return ''.join(str(v) for row in board for v in row)
def print_board(board):
    for row in board:
        print(' '.join(str(v) if v else '.' for v in row))
def candidates(board, r, c):
    used = set(board[r]) | {board[i][c] for i in range(9)}
    sr, sc = 3 * (r // 3), 3 * (c // 3)
    used |= {board[sr + i][sc + j] for i in range(3) for j in range(3)}
    return [n for n in range(1, 10) if n not in used]
def is_valid(board, r, c, num):
    for i in range(9):
        if board[r][i] == num or board[i][c] == num:
            return False
    sr, sc = 3 * (r // 3), 3 * (c // 3)
    for i in range(3):
        for j in range(3):
            if board[sr + i][sc + j] == num:
                return False
    return True
class SearchLimit(Exception):
    pass
print('Example puzzle:')
print_board(string_to_board(sample.loc[0, 'puzzle']))

### What does our sample look like?

The left chart shows how many clues the puzzles in the 100,000-row pool have. The right chart shows the 30 puzzles we actually test, coloured by difficulty group.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.histplot(pool['clues'], bins=range(pool['clues'].min(), pool['clues'].max() + 2), color='#457b9d', ax=axes[0])
axes[0].set_title('Clues per puzzle (first 100,000 rows)')
axes[0].set_xlabel('Number of clues')
sns.histplot(data=sample, x='clues', hue='level', hue_order=['Hard', 'Medium', 'Easy'], multiple='stack', palette=['#e76f51', '#e9c46a', '#2a9d8f'], binwidth=1, ax=axes[1])
axes[1].set_title('Puzzles used in the experiment')
axes[1].set_xlabel('Number of clues')
plt.tight_layout()
plt.show()

## 3. The three algorithms

All three solvers take a 9x9 board (0 means empty), get the same time limit, and return the solved board or `None`, along with a few counters. Each solver counts a `node` in its own way, so node counts are useful for comparing within one algorithm but not as a perfectly fair comparison across all three.

### 3.1 Backtracking

Backtracking is the classic CSP approach. It scans the board from the top left, finds the first empty cell, tries digits 1 to 9, and keeps only the ones that do not clash with the row, column or 3x3 box. If a cell has no valid digit left, it undoes the last move and tries something else.

* **Complete:** if a solution exists, it will find it.
* **Weakness:** it always fills cells in the same fixed order, so on puzzles with few clues it can walk into huge dead ends before noticing the mistake.
* **Node:** one digit placement.

In [ ]:
def solve_backtracking(puzzle, time_limit=TIME_LIMIT):
    board = [row[:] for row in puzzle]
    stats = {'nodes': 0}
    deadline = time.perf_counter() + time_limit
    def search():
        for r in range(9):
            for c in range(9):
                if board[r][c] == 0:
                    for num in range(1, 10):
                        if is_valid(board, r, c, num):
                            stats['nodes'] += 1
                            if stats['nodes'] % 500 == 0 and time.perf_counter() > deadline:
                                raise SearchLimit
                            board[r][c] = num
                            if search():
                                return True
                            board[r][c] = 0
                    return False
        return True
    try:
        solved = search()
    except SearchLimit:
        solved = False
    return (board if solved else None), stats

### 3.2 A\* Search

Here a state is a partly filled board and one move fills one cell, so each move costs 1. The heuristic is the number of empty cells left. That is admissible, because every empty cell needs at least one move.

Two extra ideas keep the search small:

* **Most constrained cell:** the next move always fills the empty cell with the fewest legal digits (the MRV idea from our CSP lectures).
* **Dead-end pruning:** if any empty cell has no legal digit, that state is thrown away immediately.

One honest detail worth knowing for the poster: since every move costs 1 and removes exactly one empty cell, `f = g + h` is the same for every state. The priority queue therefore breaks ties by preferring the deeper state, so this A\* behaves like a guided best-first search that dives deep along the most constrained choices. The strength comes from the cell-selection rule and the pruning.

* **Complete:** yes, it can reach any solution.
* **Node:** one state taken out of the priority queue.

In [ ]:
def count_empty(board):
    return sum(row.count(0) for row in board)
def most_constrained_cell(board):
    best, options = None, None
    for r in range(9):
        for c in range(9):
            if board[r][c] == 0:
                found = candidates(board, r, c)
                if not found:
                    return None, []
                if options is None or len(found) < len(options):
                    best, options = (r, c), found
    return best, options
def solve_a_star(puzzle, time_limit=TIME_LIMIT):
    tie = itertools.count()
    start = [row[:] for row in puzzle]
    frontier = [(count_empty(start), 0, next(tie), start)]
    stats = {'nodes': 0, 'max_frontier': 1}
    deadline = time.perf_counter() + time_limit
    while frontier:
        _, neg_g, _, state = heapq.heappop(frontier)
        stats['nodes'] += 1
        if time.perf_counter() > deadline:
            return None, stats
        cell, options = most_constrained_cell(state)
        if options is None:
            return state, stats
        if not options:
            continue
        g = -neg_g
        r, c = cell
        for num in options:
            child = [row[:] for row in state]
            child[r][c] = num
            heapq.heappush(frontier, (g + 1 + count_empty(child), -(g + 1), next(tie), child))
        stats['max_frontier'] = max(stats['max_frontier'], len(frontier))
    return None, stats

### 3.3 Hill Climbing

Hill climbing works in a completely different way. It never builds a solution step by step. Instead it starts with a full board where every 3x3 box already holds the digits 1 to 9 once (the empty cells are filled randomly), and then it tries to fix the rows and columns.

* **Cost:** the number of repeated digits in rows and columns. A cost of 0 means the puzzle is solved.
* **Move:** swap two non-given cells inside the same box, so the boxes stay valid.
* **Strategy:** always take the best swap. Sideways moves are allowed up to a limit, and when it gets stuck in a local minimum it restarts from a new random board.
* **Not complete:** it can get stuck forever on a plateau or in a local minimum, and nothing guarantees it will ever reach cost 0.
* **Node:** one swap that was evaluated.

In [ ]:
def fill_boxes(puzzle):
    board = [row[:] for row in puzzle]
    free = []
    for br in range(0, 9, 3):
        for bc in range(0, 9, 3):
            cells = [(br + i, bc + j) for i in range(3) for j in range(3)]
            present = {board[r][c] for r, c in cells if board[r][c]}
            missing = [n for n in range(1, 10) if n not in present]
            random.shuffle(missing)
            empties = [(r, c) for r, c in cells if board[r][c] == 0]
            for (r, c), value in zip(empties, missing):
                board[r][c] = value
            free.append(empties)
    return board, free
def line_cost(board, rows, cols):
    return sum(9 - len(set(board[r])) for r in rows) + sum(9 - len({board[i][c] for i in range(9)}) for c in cols)
def solve_hill_climbing(puzzle, max_restarts=200, max_steps=300, max_sideways=30, time_limit=TIME_LIMIT):
    deadline = time.perf_counter() + time_limit
    stats = {'nodes': 0, 'restarts': 0, 'final_conflicts': None}
    best_seen = None
    for _ in range(max_restarts):
        stats['restarts'] += 1
        board, free = fill_boxes(puzzle)
        cost = line_cost(board, range(9), range(9))
        sideways = 0
        for _ in range(max_steps):
            if cost == 0:
                break
            if time.perf_counter() > deadline:
                return None, stats
            best_delta, best_moves = None, []
            for cells in free:
                for a in range(len(cells)):
                    for b in range(a + 1, len(cells)):
                        (r1, c1), (r2, c2) = cells[a], cells[b]
                        if board[r1][c1] == board[r2][c2]:
                            continue
                        rows, cols = {r1, r2}, {c1, c2}
                        before = line_cost(board, rows, cols)
                        board[r1][c1], board[r2][c2] = board[r2][c2], board[r1][c1]
                        after = line_cost(board, rows, cols)
                        board[r1][c1], board[r2][c2] = board[r2][c2], board[r1][c1]
                        stats['nodes'] += 1
                        delta = after - before
                        if best_delta is None or delta < best_delta:
                            best_delta, best_moves = delta, [(r1, c1, r2, c2)]
                        elif delta == best_delta:
                            best_moves.append((r1, c1, r2, c2))
            if best_delta is None or best_delta > 0:
                break
            sideways = sideways + 1 if best_delta == 0 else 0
            if sideways > max_sideways:
                break
            r1, c1, r2, c2 = random.choice(best_moves)
            board[r1][c1], board[r2][c2] = board[r2][c2], board[r1][c1]
            cost += best_delta
        if cost == 0:
            stats['final_conflicts'] = 0
            return board, stats
        best_seen = cost if best_seen is None else min(best_seen, cost)
        stats['final_conflicts'] = best_seen
    return None, stats

## 4. Running the experiment

Every algorithm gets the same puzzles and the same time limit. A run only counts as solved if the board it returns matches the official solution from the dataset. Hill climbing uses random restarts, so the seed is fixed to make the run repeatable.

This is the slow part. With 30 puzzles it can take several minutes, mostly because hill climbing and the hardest puzzles use up their full time limit.

In [ ]:
def run_benchmark(solvers):
    rows = []
    for idx, item in sample.iterrows():
        board = string_to_board(item['puzzle'])
        for name, solver in solvers.items():
            start = time.perf_counter()
            solution, stats = solver(board)
            elapsed = time.perf_counter() - start
            rows.append({'puzzle': idx, 'clues': item['clues'], 'level': item['level'], 'algorithm': name,
                         'correct': solution is not None and board_to_string(solution) == item['solution'],
                         'time_s': elapsed, 'nodes': stats['nodes'], 'conflicts': stats.get('final_conflicts')})
        print(f'puzzle {idx + 1}/{len(sample)} finished')
    return pd.DataFrame(rows)
random.seed(42)
results = run_benchmark({'Backtracking': solve_backtracking, 'A*': solve_a_star, 'Hill Climbing': solve_hill_climbing})

## 5. Results

### 5.1 Summary table

* `solved_pct`: share of puzzles solved correctly within the time limit
* `avg_time_s` and `median_time_s`: time per puzzle (unsolved runs count as the full time limit)
* `avg_nodes`: average search effort per puzzle

The table is sorted so that the algorithm with the highest success rate comes first, and the average time breaks ties.

In [ ]:
summary = results.groupby('algorithm').agg(solved_pct=('correct', 'mean'), avg_time_s=('time_s', 'mean'),
                                           median_time_s=('time_s', 'median'), avg_nodes=('nodes', 'mean'))
summary['solved_pct'] = summary['solved_pct'] * 100
summary = summary.sort_values(['solved_pct', 'avg_time_s'], ascending=[False, True]).round(3)
order = summary.index.tolist()
summary

### 5.2 Success rate and average time

Left: how many puzzles each algorithm solved correctly. Right: the average time per puzzle on a log scale, so very fast and very slow algorithms can sit on the same chart.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes[0].bar(order, summary['solved_pct'], color=[palette[a] for a in order])
axes[0].bar_label(axes[0].containers[0], fmt='%.0f%%', fontsize=14)
axes[0].set_ylim(0, 115)
axes[0].set_ylabel('Puzzles solved (%)')
axes[0].set_title('Success rate')
axes[1].bar(order, summary['avg_time_s'], color=[palette[a] for a in order])
axes[1].bar_label(axes[1].containers[0], labels=[f'{v:.3g} s' for v in summary['avg_time_s']], fontsize=14)
axes[1].set_yscale('log')
axes[1].set_ylabel('Average time per puzzle (log scale)')
axes[1].set_title('Speed')
plt.tight_layout()
plt.show()

### 5.3 Time spread across puzzles

An average can hide a lot. This box plot shows the time for every single puzzle. The dashed line is the time limit, so any dot sitting on it means that run gave up. A short, low box means the algorithm is fast and also predictable.

In [ ]:
plt.figure(figsize=(11, 6))
sns.boxplot(data=results, x='algorithm', y='time_s', order=order, hue='algorithm', palette=palette, legend=False, width=0.5)
sns.stripplot(data=results, x='algorithm', y='time_s', order=order, color='black', alpha=0.5, size=5)
plt.axhline(TIME_LIMIT, color='red', linestyle='--', linewidth=1.5, label='Time limit')
plt.yscale('log')
plt.xlabel('')
plt.ylabel('Time per puzzle in seconds (log scale)')
plt.title('Time distribution over all tested puzzles')
plt.legend()
plt.tight_layout()
plt.show()

### 5.4 Does difficulty matter?

Puzzles are grouped by how many clues they give. A robust algorithm should keep its success rate even on the Hard group.

In [ ]:
by_level = results.pivot_table(index='level', columns='algorithm', values='correct', aggfunc='mean', observed=True) * 100
by_level = by_level.reindex(['Easy', 'Medium', 'Hard'])[order]
ax = by_level.plot(kind='bar', figsize=(11, 6), color=[palette[a] for a in order], width=0.75)
for container in ax.containers:
    ax.bar_label(container, fmt='%.0f%%', fontsize=11)
ax.set_ylim(0, 115)
ax.set_xlabel('Difficulty group')
ax.set_ylabel('Puzzles solved (%)')
ax.set_title('Success rate by difficulty')
plt.xticks(rotation=0)
plt.legend(title='', loc='upper center', bbox_to_anchor=(0.5, -0.18), ncol=3)
plt.tight_layout()
plt.show()

### 5.5 Time against number of clues

Each dot is one puzzle. If a dot climbs as the clue count falls, that algorithm struggles with sparse puzzles.

In [ ]:
plt.figure(figsize=(12, 6))
sns.scatterplot(data=results, x='clues', y='time_s', hue='algorithm', hue_order=order, palette=palette, s=130, alpha=0.8)
plt.axhline(TIME_LIMIT, color='red', linestyle='--', linewidth=1.5)
plt.yscale('log')
plt.xlabel('Number of clues given')
plt.ylabel('Time per puzzle in seconds (log scale)')
plt.title('Solving time vs puzzle difficulty')
plt.legend(title='', loc='upper left', bbox_to_anchor=(1.02, 1))
plt.tight_layout()
plt.show()

### 5.6 Search effort

The number of nodes shows how much work an algorithm does. Green dots are runs that solved the puzzle, red dots are runs that failed. Remember that a node means something slightly different in each algorithm, so compare the dots inside one column first.

In [ ]:
plt.figure(figsize=(11, 6))
sns.stripplot(data=results, x='algorithm', y='nodes', order=order, hue='correct', hue_order=[True, False], palette={True: '#2a9d8f', False: '#e63946'}, size=9, alpha=0.8)
plt.yscale('log')
plt.xlabel('')
plt.ylabel('Nodes explored (log scale)')
plt.title('Search effort per run')
plt.legend(title='Solved', loc='upper left', bbox_to_anchor=(1.02, 1))
plt.tight_layout()
plt.show()

### 5.7 How close does Hill Climbing get?

When hill climbing fails, it is useful to know how badly. This chart shows the lowest number of row and column conflicts it reached on each puzzle. A green dot at zero means it solved the puzzle. Anything above zero is a board that looks almost right but is still not a valid solution.

In [ ]:
hc = results[results.algorithm == 'Hill Climbing'].dropna(subset=['conflicts']).sort_values('clues')
plt.figure(figsize=(13, 5))
plt.vlines(range(len(hc)), 0, hc['conflicts'], color='#cccccc', linewidth=3)
plt.scatter(range(len(hc)), hc['conflicts'], s=170, zorder=3, color=['#2a9d8f' if v == 0 else '#e76f51' for v in hc['conflicts']])
plt.xticks(range(len(hc)), hc['clues'].astype(int), rotation=90, fontsize=11)
plt.xlabel('Puzzle (labelled by number of clues)')
plt.ylabel('Best conflict count reached')
plt.title('Hill Climbing: remaining conflicts per puzzle (green = solved)')
plt.tight_layout()
plt.show()

## 6. Extra check: is A\* winning because of A\* or because of MRV?

A fair question from anyone looking at the poster: the A\* solver chooses the most constrained cell, while plain backtracking just takes the first empty cell. So is the advantage coming from the search method itself or only from that smarter cell choice?

To find out, we run a fourth solver, backtracking with the same most-constrained-cell rule, on the same puzzles.

In [ ]:
def solve_backtracking_mrv(puzzle, time_limit=TIME_LIMIT):
    board = [row[:] for row in puzzle]
    stats = {'nodes': 0}
    deadline = time.perf_counter() + time_limit
    def search():
        cell, options = most_constrained_cell(board)
        if options is None:
            return True
        if not options:
            return False
        r, c = cell
        for num in options:
            stats['nodes'] += 1
            if stats['nodes'] % 500 == 0 and time.perf_counter() > deadline:
                raise SearchLimit
            board[r][c] = num
            if search():
                return True
            board[r][c] = 0
        return False
    try:
        solved = search()
    except SearchLimit:
        solved = False
    return (board if solved else None), stats
extra = run_benchmark({'Backtracking + MRV': solve_backtracking_mrv})
ablation = pd.concat([results[results.algorithm.isin(['Backtracking', 'A*'])], extra])
ablation_summary = ablation.groupby('algorithm').agg(solved_pct=('correct', 'mean'), avg_time_s=('time_s', 'mean'), avg_nodes=('nodes', 'mean'))
ablation_summary['solved_pct'] = ablation_summary['solved_pct'] * 100
ablation_summary = ablation_summary.reindex(['Backtracking', 'Backtracking + MRV', 'A*']).round(3)
ablation_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
names = ablation_summary.index.tolist()
labels = [n.replace(' + ', '\n+ ') for n in names]
axes[0].bar(labels, ablation_summary['solved_pct'], color=[palette[n] for n in names])
axes[0].bar_label(axes[0].containers[0], fmt='%.0f%%', fontsize=14)
axes[0].set_ylim(0, 115)
axes[0].set_ylabel('Puzzles solved (%)')
axes[0].set_title('Success rate')
axes[1].bar(labels, ablation_summary['avg_time_s'], color=[palette[n] for n in names])
axes[1].bar_label(axes[1].containers[0], labels=[f'{v:.3g} s' for v in ablation_summary['avg_time_s']], fontsize=14)
axes[1].set_yscale('log')
axes[1].set_ylabel('Average time per puzzle (log scale)')
axes[1].set_title('Speed')
plt.tight_layout()
plt.show()

## 7. Verdict

The cell below picks the winner straight from the numbers: highest success rate first, then lowest average time.

In [ ]:
winner = summary.index[0]
print(f'Best algorithm on this run: {winner}')
print(f"Solved {summary.loc[winner, 'solved_pct']:.1f}% of puzzles in {summary.loc[winner, 'avg_time_s']:.3f} s on average")
print()
for rank, name in enumerate(summary.index, start=1):
    print(f"{rank}. {name}: {summary.loc[name, 'solved_pct']:.1f}% solved, {summary.loc[name, 'avg_time_s']:.3f} s average")

### Why the ranking comes out this way

**Completeness decides most of it.** Sudoku is a constraint satisfaction problem with a single correct answer. Backtracking and A\* are both complete, so given enough time they cannot miss the solution. Hill climbing is a local search that only improves a guess, so it can get trapped in a local minimum or on a plateau and never find the answer, no matter how many restarts it gets. Sudoku has many such traps, which is why it is expected to be the least dependable of the three, especially on sparse puzzles.

**Pruning decides the rest.** Plain backtracking checks cells in a fixed order and only notices a mistake when it is already deep in the tree. A\* in our setup always expands the most constrained cell and drops any state with a cell that has no legal digit. Failing early is what keeps it fast on puzzles with few clues. The extra check in Section 6 shows how much of the gain comes from this cell-selection rule and how much comes from the A\* machinery itself.

**Memory cost.** A\* stores many boards in its priority queue, so it uses more memory than backtracking, which only keeps one board. On a 9x9 grid this is small, but it is a fair trade-off to mention.

### Points for the poster

* The best algorithm is the one the table in Section 7 prints, and the graphs above are the evidence for it.
* Backtracking is the simplest correct baseline and works well on easy puzzles.
* If Backtracking + MRV ends up level with A\* in Section 6, say so openly: it means the smart cell choice does the heavy lifting, and A\* adds a clean framework around it.
* Hill climbing is a good fit for problems where any decent answer is acceptable, and a poor fit for Sudoku, where only the exact solution counts.
* Results depend on the sample and on the time limit, so state `N_PUZZLES` and `TIME_LIMIT` on the poster next to the numbers.

### Limits of this study

* Only 30 puzzles from the first 100,000 rows were tested, so a bigger sample would give steadier percentages.
* Timings are for pure Python on Colab and will shift a little between runs and machines.
* Hill climbing is random, so a different seed can change its numbers slightly.